# 02b — Khôi phục các bài Lao Động lưu nhầm trang cookie
Chạy trên **CPU** sau notebook 01 và 02 của Stage A. Notebook đọc raw `stage-a-v2`, chỉ lấy lại những bản ghi Lao Động có trang cookie D1N, giữ nguyên các ID còn lại, rồi xây một corpus mới đủ 1.000 dòng. Raw và build cũ không bị sửa.

Chỉ kết quả được adapter xác nhận có đúng article ID, tiêu đề và thân bài mới thay thế raw cũ. Nếu trang tiếp tục lỗi, bản ghi cũ được giữ để báo cáo parse failure.


In [ ]:
import importlib.util, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = Path('/content/drive/MyDrive/VietMedBridge/data')
WORK_DIR = Path('/content/vmb_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
REPO_URL = 'https://github.com/Platypus27-coder/VietMedBridge.git'
CODE_REF = 'feature/laodong-extractor'
CHECKOUT = Path('/content/VietMedBridge')
if 'vietmedbridge' in sys.modules:
    raise RuntimeError('Đã import package trong runtime này. Restart session rồi chạy lại 02b.')
if not (CHECKOUT / '.git').exists():
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(CHECKOUT)], check=True)
if subprocess.check_output(['git', '-C', str(CHECKOUT), 'remote', 'get-url', 'origin'], text=True).strip() != REPO_URL:
    raise RuntimeError('Checkout không thuộc repo VietMedBridge')
subprocess.run(['git', '-C', str(CHECKOUT), 'fetch', '--depth', '1', 'origin', CODE_REF], check=True)
subprocess.run(['git', '-C', str(CHECKOUT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
CODE_COMMIT = subprocess.check_output(['git', '-C', str(CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[notebook]'], cwd=CHECKOUT, check=True)
sys.path.insert(0, str(CHECKOUT / 'src'))
importlib.invalidate_caches()
print('Code commit:', CODE_COMMIT)


In [ ]:
from vietmedbridge.artifacts import read_json
from vietmedbridge.crawl import CrawlConfig, read_completed_crawl
from vietmedbridge.dataset import load_snapshot, parquet_path
from vietmedbridge.parse_recovery import recover_laodong_challenges
SNAPSHOT = load_snapshot(DATA_ROOT)
PILOT = read_json(DATA_ROOT / 'reports/inventory/stage_a.json')
SOURCE_CRAWL = 'stage-a-v2'
RECOVERED_CRAWL = 'stage-a-laodong-recovered-v1'
NEW_BUILD = 'stage-a-data-v3-laodong'
BASELINE = read_completed_crawl(
    DATA_ROOT / 'crawl' / SOURCE_CRAWL,
    links_path=DATA_ROOT / PILOT['files']['stage_a_links']['path'],
    origin_corpus_sha256=SNAPSHOT['files']['links_corpus.parquet']['sha256'],
)
print('Bản ghi gốc:', BASELINE['recorded_documents'], '| HTTP ok:', BASELINE['statuses'].get('ok', 0))
RECOVERY = await recover_laodong_challenges(
    DATA_ROOT / 'crawl' / SOURCE_CRAWL, DATA_ROOT / 'crawl',
    run_name=RECOVERED_CRAWL, work_dir=WORK_DIR,
    config=CrawlConfig(concurrency=2, per_host_delay=2.0, attempts=2),
)
print(json.dumps(RECOVERY, ensure_ascii=False, indent=2))


In [ ]:
from vietmedbridge.build import build_corpus
from vietmedbridge.chunks import ChunkConfig, load_bge_tokenizer
from vietmedbridge.health import health_report
TOKENIZER_LOCK = read_json(DATA_ROOT / 'tokenizer_lock.json')
TOKENIZER, TOKENIZER_SPEC = load_bge_tokenizer(TOKENIZER_LOCK['revision'])
PIPELINE_CONFIG = json.loads((CHECKOUT / 'configs/data_pipeline.json').read_text())
BUILD = build_corpus(
    DATA_ROOT / 'crawl' / RECOVERED_CRAWL, DATA_ROOT / 'processed',
    TOKENIZER, TOKENIZER_SPEC, run_name=NEW_BUILD,
    config=ChunkConfig(**PIPELINE_CONFIG['chunking']), work_dir=WORK_DIR,
    official_links=parquet_path(DATA_ROOT, 'links_corpus.parquet'),
)
print('Build counts:', json.dumps(BUILD['counts'], ensure_ascii=False, indent=2))
HEALTH = health_report(
    DATA_ROOT / 'processed' / NEW_BUILD,
    official_links=parquet_path(DATA_ROOT, 'links_corpus.parquet'),
    crawl_dir=DATA_ROOT / 'crawl' / RECOVERED_CRAWL,
    work_dir=WORK_DIR, audit_size=PIPELINE_CONFIG['audit_size'],
)
print('Coverage:', json.dumps(HEALTH['coverage'], ensure_ascii=False, indent=2))
print('Lao Động:', [row for row in HEALTH['by_domain'] if 'laodong.vn' in row.get('domain', '')])


So sánh `recovered_articles`, `documents` và `parse_failed` với baseline 766/151. Sau đó có thể mở notebook 03, đặt `CODE_REVISION = 'feature/laodong-extractor'`, `CRAWL_RUN = 'stage-a-laodong-recovered-v1'`, `BUILD_RUN = 'stage-a-data-v3-laodong'` để kiểm tra/freeze bản mới.
